In [ ]:
%load_ext autoreload
%autoreload 2


# nlpie Normalization Toolkit

This notebook demonstrates all five embedding normalization routines provided by `nlpie`, along with correlation analysis, hubness explanation, and visualizations.

**Normalization functions covered:**
- `mean_center` — subtract the column-wise mean
- `l2_normalize_rows` — scale each row to unit Euclidean norm
- `standardize_columns` — zero mean, unit variance per column (z-score)
- `whiten_pca` — PCA whitening (decorrelate + scale)
- `remove_top_principal_components` — debias by nullifying top PC directions

In [ ]:
import sys
import math
import numpy as np

sys.path.insert(0, "..")
sys.path.insert(0, "../../python")

from nlpie import (
    EmbeddingPreprocessor,
    FitStats,
    WhitenModel,
    mean_center,
    l2_normalize_rows,
    standardize_columns,
    whiten_pca,
    remove_top_principal_components,
    cosine_similarity_matrix,
    cosine_similarity_matrix_stats,
    pearson_correlation,
    spearman_correlation,
    compute_hubness,
    effective_rank,
    similarity_to_global_mean,
    explain_hubness,
    plot_hubness_histogram,
    plot_similarity_distribution,
    plot_similarity_to_mean,
    plot_similarity_heatmap,
    plot_eigenvalue_scree,
    plot_embedding_scatter,
)

In [ ]:
# Configure Plotly for Jupyter rendering
import plotly.io as pio
pio.renderers.default = "notebook_connected"


## 1. Generate Synthetic Data

We create a random embedding matrix — 500 samples, 32 dimensions — with a small cluster structure.

In [ ]:
np.random.seed(42)
n, d = 500, 32
labels = np.random.randint(0, 4, n)
centers = np.random.randn(4, d) * 3.0
embeddings = centers[labels] + np.random.randn(n, d) * 0.8

print(f"Shape: {embeddings.shape}  dtype: {embeddings.dtype}")
print(f"Row norms: mean={np.linalg.norm(embeddings, axis=1).mean():.3f}  std={np.linalg.norm(embeddings, axis=1).std():.3f}")
print(f"Column means: min={embeddings.mean(axis=0).min():.3f}  max={embeddings.mean(axis=0).max():.3f}")
print(f"Column stds:  min={embeddings.std(axis=0).min():.3f}  max={embeddings.std(axis=0).max():.3f}")

## 2. Correlation Analysis

Before normalizing, examine pairwise correlations between embedding vectors.

In [ ]:
p = pearson_correlation(embeddings[0], embeddings[1])
s = spearman_correlation(embeddings[0], embeddings[2])
print(f"Pearson r (point 0 vs 1): {p:.4f}")
print(f"Spearman ρ (point 0 vs 2): {s:.4f}")

## 3. Functional API vs Preprocessor API

`nlpie` offers two styles:
- **Free functions** — one-shot calls.
- **`EmbeddingPreprocessor` object** — holds a persistent `eps` value.

Both call the same Rust backend.

### 3a. Mean Center

Subtracts the column-wise mean so every column has zero mean.

In [ ]:
centered, stats = mean_center(embeddings)
centered_arr = np.array(centered)

print("--- Mean Center ---")
print(f"Column means after: min={centered_arr.mean(axis=0).min():.6f}  max={centered_arr.mean(axis=0).max():.6f}")
print(f"FitStats.mean[:5]  = {stats.mean[:5]}")
print(f"FitStats.std       = {stats.std}")

### 3b. L2 Normalize Rows

Scales each row to unit Euclidean norm.

In [ ]:
normed = l2_normalize_rows(embeddings)
normed_arr = np.array(normed)
row_norms = np.linalg.norm(normed_arr, axis=1)

print("--- L2 Normalize Rows ---")
print(f"Row norms: mean={row_norms.mean():.6f}  std={row_norms.std():.6f}  max={row_norms.max():.6f}")

### 3c. Standardize Columns (Z-Score)

Zero mean **and** unit variance per column.

In [ ]:
standardized, stats_z = standardize_columns(embeddings)
std_arr = np.array(standardized)

print("--- Standardize Columns ---")
print(f"Column means: min={std_arr.mean(axis=0).min():.6f}  max={std_arr.mean(axis=0).max():.6f}")
print(f"Column stds:  min={std_arr.std(axis=0).min():.6f}  max={std_arr.std(axis=0).max():.6f}")
print(f"FitStats.mean[:3] = {stats_z.mean[:3]}")
print(f"FitStats.std[:3]  = {stats_z.std[:3]}  (epsilon clipped)")

### 3d. PCA Whitening

Decorrelates features via PCA and scales components to unit variance.

In [ ]:
whitened_all, wm = whiten_pca(embeddings)
w_arr = np.array(whitened_all)

print("--- PCA Whitening (all components) ---")
print(f"Shape: {w_arr.shape}")
print(f"Column means: min={w_arr.mean(axis=0).min():.6f}  max={w_arr.mean(axis=0).max():.6f}")
print(f"Column stds:  min={w_arr.std(axis=0).min():.6f}  max={w_arr.std(axis=0).max():.6f}")
print(f"Projection shape: {len(wm.projection)} x {len(wm.projection[0])}")
print(f"Eigenvalues[:5]:   {wm.eigenvalues[:5]}")

In [ ]:
whitened_8, wm8 = whiten_pca(embeddings, n_components=8)
w8_arr = np.array(whitened_8)

print("--- PCA Whitening (8 components) ---")
print(f"Shape: {w8_arr.shape}")
print(f"Eigenvalues: {wm8.eigenvalues}")

### 3e. Remove Top Principal Components (Debiasing)

Removes the top-`k` principal components to nullify dominant variation directions.

In [ ]:
debias_3 = remove_top_principal_components(embeddings, n_components=3)
db3_arr = np.array(debias_3)

print("--- Remove Top 3 PCs ---")
print(f"Shape: {db3_arr.shape}")
print(f"Effective rank before: {np.linalg.matrix_rank(embeddings)}")
print(f"Effective rank after:  {np.linalg.matrix_rank(db3_arr)}")
print(f"Column means: min={db3_arr.mean(axis=0).min():.6f}  max={db3_arr.mean(axis=0).max():.6f}")

## 4. Eigenvalue Scree Plot

Visualize the eigenvalue spectrum from PCA whitening.

In [ ]:
fig = plot_eigenvalue_scree(wm.eigenvalues)
fig.show()

## 5. Chaining — Using the Preprocessor Object

When you need to apply multiple normalization steps, the `EmbeddingPreprocessor` avoids repeating `eps`.

In [ ]:
prep = EmbeddingPreprocessor()

c, _ = prep.mean_center(embeddings)
cn = prep.l2_normalize_rows(c)

cn_arr = np.array(cn)
print("--- Chained: mean_center → l2_normalize_rows ---")
print(f"Row norms after chain: mean={np.linalg.norm(cn_arr, axis=1).mean():.6f}")
print(f"Column means after chain: min={cn_arr.mean(axis=0).min():.6f}")

## 6. Visualizations

After preprocessing, visualize the embedding space.

In [ ]:
# Cosine similarity matrix stats
sim_mat, mean_sim, std_sim, min_sim, max_sim = cosine_similarity_matrix_stats(cn_arr)
print(f"Pairwise cosine similarity — mean={mean_sim:.4f}  std={std_sim:.4f}")
print(f"  min={min_sim:.4f}  max={max_sim:.4f}")

In [ ]:
# Similarity distribution
n_vals = len(sim_mat)
vals = [sim_mat[i][j] for i in range(n_vals) for j in range(i + 1, n_vals)]
plot_similarity_distribution(vals, mean_sim, std_sim).show()

In [ ]:
# Similarity heatmap (labeled by cluster)
plot_similarity_heatmap(sim_mat, labels=labels.tolist()).show()

In [ ]:
# Similarity-to-mean chart
sims = similarity_to_global_mean(cn_arr)
plot_similarity_to_mean(sims).show()

In [ ]:
# 2D scatter of the first 2 PCA components
pca_2d = whiten_pca(cn_arr, n_components=2)[0]
pca_arr = np.array(pca_2d)
plot_embedding_scatter(pca_arr[:, 0], pca_arr[:, 1], labels=labels.tolist(),
    title="PCA 2D Projection (after preprocessing)").show()

## 7. Hubness Analysis

Compute hubness and get an explanation.

In [ ]:
counts, skewness = compute_hubness(cn_arr, k=5)
print(f"Hubness skewness: {skewness:.4f}")

fig = plot_hubness_histogram(counts, skewness, k=5)
fig.show()

In [ ]:
explanation = explain_hubness(counts, skewness, k=5, top_n=3)
print(f"Severity: {explanation.severity}")
print(f"Summary: {explanation.summary}")
print(f"Top hubs: {[(h.index, h.count) for h in explanation.top_hubs]}")
print(f"Interpretation: {explanation.interpretation}")

## 8. Summary

| Function | Return | Effect |
|---|---|---|
| `mean_center` | `(matrix, FitStats)` | Each column → mean 0 |
| `l2_normalize_rows` | `matrix` | Each row → unit norm |
| `standardize_columns` | `(matrix, FitStats)` | Each column → mean 0, std 1 |
| `whiten_pca` | `(matrix, WhitenModel)` | Decorrelate + unit variance |
| `remove_top_principal_components` | `matrix` | Nullify top-k PC directions |

Additional features demonstrated:
- Pearson / Spearman correlation
- Cosine similarity matrix + stats
- Eigenvalue scree plot
- Hubness analysis + explanation
- Similarity heatmap, distribution, and scatter plots